# EDA: Evaluation Corpus (5,005 Letter Texts)

**Purpose**: Explore the unified evaluation corpus for LLM baseline predictions

**Data**: 
- Word-level: `v_1/data/evaluation_corpora/unified_3groups_akkadian_letters.parquet`
- Text-level: `v_1/data/evaluation_corpora/texts_for_evaluation.parquet`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
# Load word-level data
word_df = pd.read_parquet('../data/evaluation_corpora/unified_3groups_akkadian_letters.parquet')
print(f"Word-level data: {len(word_df):,} rows")
print(f"Unique texts: {word_df['fragment_id'].nunique():,}")

# Load text-level data
text_df = pd.read_parquet('../data/evaluation_corpora/texts_for_evaluation.parquet')
print(f"\nText-level data: {len(text_df):,} rows")

## 2. Word-Level Data Overview

In [ ]:
# Show columns
print("Columns:")
print(list(word_df.columns))

In [ ]:
# Data types and info
word_df.info()

In [ ]:
# Sample rows
print("Sample word-level data:")
word_df.head(20)

## 3. Text-Level Data Overview

In [ ]:
# Show columns
print("Columns:")
print(list(text_df.columns))

In [ ]:
# Data types and info
text_df.info()

In [ ]:
# Basic statistics
text_df.describe()

In [ ]:
# Sample texts (showing first 500 chars of full_text)
sample = text_df[['fragment_id', 'full_text', 'word_count', 'line_count', 
                   'temporal_group', 'period', 'domain_standard', 'corpus_source']].head(10)
sample['full_text_preview'] = sample['full_text'].str[:200] + '...'
sample.drop('full_text', axis=1)

## 4. Distribution by Temporal Group & Period

In [ ]:
# Count by temporal group
print("Distribution by Temporal Group:")
print(text_df['temporal_group'].value_counts().sort_index())
print(f"\nTotal: {len(text_df):,} texts")

In [ ]:
# Count by period
print("Distribution by Period:")
print(text_df['period'].value_counts())

In [ ]:
# Visualize temporal distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By temporal group
text_df['temporal_group'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribution by Temporal Group', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Temporal Group')
axes[0].set_ylabel('Number of Texts')
axes[0].tick_params(axis='x', rotation=0)

# By period
period_order = ['Old Babylonian', 'Neo-Assyrian', 'Late Babylonian']
text_df['period'].value_counts().reindex(period_order).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Distribution by Period', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Number of Texts')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Distribution by Corpus Source

In [ ]:
# Count by corpus source
print("Distribution by Corpus Source:")
print(text_df['corpus_source'].value_counts())

In [ ]:
# Cross-tab: corpus source vs period
pd.crosstab(text_df['corpus_source'], text_df['period'], margins=True)

## 6. Domain Analysis

In [ ]:
# Domain standard distribution
print("Distribution by Domain (Standard):")
print(text_df['domain_standard'].value_counts())

In [ ]:
# Domain finegrained distribution
print("Distribution by Domain (Fine-grained):")
print(text_df['domain_finegrained'].value_counts())

In [ ]:
# Domain by period
pd.crosstab(text_df['period'], text_df['domain_finegrained'], margins=True)

## 7. Place of Discovery

In [ ]:
# Top places by temporal group
print("Top Places of Discovery by Temporal Group:\n")
for group in sorted(text_df['temporal_group'].unique()):
    print(f"{group}:")
    places = text_df[text_df['temporal_group'] == group]['place_discovery'].value_counts().head(10)
    print(places)
    print()

In [ ]:
# Visualize top places per period
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

periods = ['Old Babylonian', 'Neo-Assyrian', 'Late Babylonian']
for i, period in enumerate(periods):
    subset = text_df[text_df['period'] == period]
    top_places = subset['place_discovery'].value_counts().head(10)
    top_places.plot(kind='barh', ax=axes[i], color='teal')
    axes[i].set_title(f'{period} - Top 10 Places of Discovery', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Number of Texts')
    
plt.tight_layout()
plt.show()

## 8. Text Length Statistics

In [ ]:
# Overall statistics
print("Text Length Statistics:")
print(f"\nWord Count:")
print(text_df['word_count'].describe())
print(f"\nLine Count:")
print(text_df['line_count'].describe())
print(f"\nCharacter Count:")
print(text_df['char_count'].describe())

In [ ]:
# Length statistics by temporal group
print("Average Text Lengths by Temporal Group:\n")
text_df.groupby('temporal_group')[['word_count', 'line_count', 'char_count']].mean().round(1)

In [ ]:
# Visualize text length distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Word count distribution
text_df.boxplot(column='word_count', by='temporal_group', ax=axes[0])
axes[0].set_title('Word Count by Temporal Group')
axes[0].set_xlabel('Temporal Group')
axes[0].set_ylabel('Word Count')
axes[0].get_figure().suptitle('')  # Remove auto title

# Line count distribution
text_df.boxplot(column='line_count', by='temporal_group', ax=axes[1])
axes[1].set_title('Line Count by Temporal Group')
axes[1].set_xlabel('Temporal Group')
axes[1].set_ylabel('Line Count')
axes[1].get_figure().suptitle('')

# Character count distribution
text_df.boxplot(column='char_count', by='temporal_group', ax=axes[2])
axes[2].set_title('Character Count by Temporal Group')
axes[2].set_xlabel('Temporal Group')
axes[2].set_ylabel('Character Count')
axes[2].get_figure().suptitle('')

plt.tight_layout()
plt.show()

## 9. Sample Texts by Period

In [ ]:
# Show one example text from each period
for period in ['Old Babylonian', 'Neo-Assyrian', 'Late Babylonian']:
    print("=" * 80)
    print(f"SAMPLE TEXT: {period}")
    print("=" * 80)
    
    sample = text_df[text_df['period'] == period].iloc[0]
    
    print(f"Fragment ID: {sample['fragment_id']}")
    print(f"Temporal Group: {sample['temporal_group']}")
    print(f"Period: {sample['period']} ({sample.get('period_approx', 'N/A')})")
    print(f"Domain: {sample['domain_standard']} / {sample.get('domain_finegrained', 'N/A')}")
    print(f"Place: {sample['place_discovery']}")
    print(f"Source: {sample['corpus_source']}")
    print(f"Length: {sample['word_count']} words, {sample['line_count']} lines, {sample['char_count']} chars")
    print(f"\nText (first 500 chars):")
    print("-" * 80)
    print(sample['full_text'][:500])
    print("...")
    print()

## 10. Data Quality Checks

In [ ]:
# Check for missing values
print("Missing values:")
print(text_df.isnull().sum())

In [ ]:
# Check for empty texts
empty_texts = text_df[text_df['full_text'].str.strip() == '']
print(f"Empty texts: {len(empty_texts)}")

zero_word = text_df[text_df['word_count'] == 0]
print(f"Zero word count: {len(zero_word)}")

In [ ]:
# Check temporal group consistency
print("Temporal Group vs Period consistency check:")
pd.crosstab(text_df['temporal_group'], text_df['period'])

In [ ]:
# Check corpus source consistency
print("Corpus Source vs Period consistency check:")
pd.crosstab(text_df['corpus_source'], text_df['period'])

## 11. Summary Statistics for LLM Evaluation

In [ ]:
# Load token statistics if available
import json

token_stats_path = Path('../data/evaluation_corpora/texts_token_stats.json')
if token_stats_path.exists():
    with open(token_stats_path, 'r') as f:
        token_stats = json.load(f)
    
    print("Token Statistics for LLM API Calls:")
    print("=" * 50)
    print(f"Total texts: {token_stats['total_texts']:,}")
    print(f"Total characters: {token_stats['total_chars']:,}")
    print(f"Estimated tokens (text): {token_stats['estimated_tokens']:,}")
    print(f"Average tokens per text: {token_stats['avg_tokens_per_text']:.1f}")
    print(f"P50 tokens per text: {token_stats['p50_tokens_per_text']:.1f}")
    print(f"P95 tokens per text: {token_stats['p95_tokens_per_text']:.1f}")
    print(f"Max tokens per text: {token_stats['max_tokens_per_text']:,}")
    print(f"Prompt template tokens: {token_stats['prompt_template_tokens']}")
    print(f"Estimated total input tokens: {token_stats['estimated_total_input_tokens']:,}")
else:
    print("Token statistics file not found. Run 01_prepare_texts.py first.")

## 12. Export Sample for Manual Inspection

In [ ]:
# Create a sample of 10 texts per period for manual review
sample_export = pd.concat([
    text_df[text_df['period'] == 'Old Babylonian'].sample(10, random_state=42),
    text_df[text_df['period'] == 'Neo-Assyrian'].sample(10, random_state=42),
    text_df[text_df['period'] == 'Late Babylonian'].sample(10, random_state=42)
])

# Select relevant columns
sample_export = sample_export[['fragment_id', 'full_text', 'word_count', 'line_count', 
                                 'temporal_group', 'period', 'period_approx',
                                 'domain_standard', 'domain_finegrained', 
                                 'place_discovery', 'corpus_source']]

print(f"Sample export: {len(sample_export)} texts (10 per period)")
sample_export.head()

In [ ]:
# Optionally save to CSV for external review
# sample_export.to_csv('../data/evaluation_corpora/sample_texts_for_review.csv', index=False)
# print("Sample exported to: sample_texts_for_review.csv")

---

## Summary

This notebook provides a **read-only EDA** of the evaluation corpus produced by the pipeline.
It does NOT modify any data files — all data preparation is done by the pipeline scripts.

**Pipeline to recreate the evaluation data:**
```bash
# Step 1: Build unified corpus from source CSVs
python v_1/src/preprocessing/06_create_test_letters_copra.py

# Step 2: Reconstruct texts + domain cleanup → parquet + JSONL
python v_1/src/evaluation/01_prepare_texts.py

# Step 3 (optional): Re-run this notebook for EDA / verification
```

**To run LLM baseline:**
```bash
export OPENROUTER_API_KEY="your-key"
python v_1/src/evaluation/02_llm_baseline.py --model gpt-oss-20b
python v_1/src/evaluation/03_aggregate_results.py
python v_1/src/evaluation/04_evaluate_baseline.py
```

## 13. Domain Label Verification

The domain label cleanup (removing Unknown/nan/Other values) is now handled
automatically by the pipeline in `01_prepare_texts.py`.

This section verifies that the loaded data is already clean.

In [ ]:
# Verify domain labels are clean (cleanup is done by 01_prepare_texts.py)
print("=" * 80)
print("DOMAIN LABEL VERIFICATION")
print("=" * 80)
print(f"Total texts: {len(text_df):,}")

checks = {
    "Unknown in domain_standard": (text_df['domain_standard'] == 'Unknown').sum(),
    "Unknown in domain_finegrained": (text_df['domain_finegrained'] == 'Unknown').sum(),
    "String 'nan' in domain_finegrained": (text_df['domain_finegrained'] == 'nan').sum(),
    "Actual NaN in domain_standard": text_df['domain_standard'].isna().sum(),
    "Actual NaN in domain_finegrained": text_df['domain_finegrained'].isna().sum(),
    "Other in domain_standard": (text_df['domain_standard'] == 'Other').sum(),
}

all_clean = True
for check_name, count in checks.items():
    status = "✅" if count == 0 else "❌"
    if count > 0:
        all_clean = False
    print(f"  {status} {check_name}: {count}")

print("\n" + "=" * 80)
if all_clean:
    print("✅ All domain labels are clean!")
else:
    print("⚠️  Issues found! Re-run: python v_1/src/evaluation/01_prepare_texts.py")
print("=" * 80)

print("\nDomain Standard:")
print(text_df['domain_standard'].value_counts(dropna=False))
print("\nDomain Finegrained:")
print(text_df['domain_finegrained'].value_counts(dropna=False))

In [ ]:
# Final corpus summary (read-only — no data modification)
print("=" * 80)
print("FINAL CORPUS SUMMARY")
print("=" * 80)
print(f"\nTotal texts: {len(text_df):,}")

print(f"\nBy Period:")
for period in ['Old Babylonian', 'Neo-Assyrian', 'Late Babylonian']:
    count = (text_df['period'] == period).sum()
    pct = 100 * count / len(text_df)
    print(f"  {period:20s}: {count:>5,} texts ({pct:5.1f}%)")

print(f"\nBy Domain (Fine-grained):")
for domain, count in text_df['domain_finegrained'].value_counts().items():
    pct = 100 * count / len(text_df)
    print(f"  {domain:30s}: {count:>5,} texts ({pct:5.1f}%)")

print(f"\nBy Corpus Source:")
for source, count in text_df['corpus_source'].value_counts().items():
    pct = 100 * count / len(text_df)
    print(f"  {source:20s}: {count:>5,} texts ({pct:5.1f}%)")

print("\n" + "=" * 80)

## 14. Reload & Verify from Disk

Reload from the parquet file on disk (produced by `01_prepare_texts.py`) and verify it matches our in-memory analysis.

In [ ]:
# Reload the cleaned data from disk to verify
verification_df = pd.read_parquet('../data/evaluation_corpora/texts_for_evaluation.parquet')

print("=" * 80)
print("FINAL VERIFICATION OF CLEANED DATA")
print("=" * 80)

# Total count
print(f"\n📊 TOTAL TEXTS: {len(verification_df):,}")
print()

# Check for Unknown values in domain_standard
unknown_standard = (verification_df['domain_standard'] == 'Unknown').sum()
print(f"❌ Unknown in domain_standard: {unknown_standard}")

# Check for Unknown values in domain_finegrained
unknown_fine = (verification_df['domain_finegrained'] == 'Unknown').sum()
print(f"❌ Unknown in domain_finegrained: {unknown_fine}")

# Check for string 'nan' in domain_finegrained
nan_string = (verification_df['domain_finegrained'] == 'nan').sum()
print(f"❌ String 'nan' in domain_finegrained: {nan_string}")

# Check for actual NaN values in domain_standard
nan_actual_standard = verification_df['domain_standard'].isna().sum()
print(f"❌ Actual NaN in domain_standard: {nan_actual_standard}")

# Check for actual NaN values in domain_finegrained
nan_actual_fine = verification_df['domain_finegrained'].isna().sum()
print(f"❌ Actual NaN in domain_finegrained: {nan_actual_fine}")

# Overall verification
total_issues = unknown_standard + unknown_fine + nan_string + nan_actual_standard + nan_actual_fine

print("\n" + "=" * 80)
if total_issues == 0:
    print("✅ VERIFICATION PASSED: No Unknown or nan values found!")
else:
    print(f"⚠️  VERIFICATION FAILED: Found {total_issues} issues")
print("=" * 80)

# Show distribution breakdown
print("\n📈 DISTRIBUTION BY PERIOD:")
print("-" * 80)
for period in ['Old Babylonian', 'Neo-Assyrian', 'Late Babylonian']:
    count = (verification_df['period'] == period).sum()
    pct = 100 * count / len(verification_df)
    print(f"  {period:20s}: {count:>5,} texts ({pct:5.1f}%)")

print(f"\n  {'TOTAL':20s}: {len(verification_df):>5,} texts (100.0%)")

print("\n📈 DISTRIBUTION BY DOMAIN (Standard):")
print("-" * 80)
for domain, count in verification_df['domain_standard'].value_counts().items():
    pct = 100 * count / len(verification_df)
    print(f"  {domain:20s}: {count:>5,} texts ({pct:5.1f}%)")

print("\n📈 DISTRIBUTION BY DOMAIN (Fine-grained) - Top 10:")
print("-" * 80)
for domain, count in verification_df['domain_finegrained'].value_counts().head(10).items():
    pct = 100 * count / len(verification_df)
    print(f"  {domain:30s}: {count:>5,} texts ({pct:5.1f}%)")

print("\n" + "=" * 80)
print(f"✅ FINAL CORPUS: {len(verification_df):,} clean Akkadian letter texts")
print("=" * 80)